In [3]:
import numpy as np
import pandas as pd
import os
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt


__mapping for categorical data conversion from orig mental health dataset__
mapGender = {'Male':0, 'Female':1, 'Other':2}  
mapMarital = {'Single':0, 'Divorced':1, 'Married':2}  
mapEdu = {'High School':0, 'Bachelor':1, 'Master':2, 'PhD':3}  
mapEmpl = {'Student':0, 'Unemployed':1, 'Employed':2, 'Self-Employed':3}  
maps = [mapGender, mapMarital, mapEdu, mapEmpl]  

In [4]:
mapGender = {'Male':0, 'Female':1, 'Other':2}
mapGenre = {"Mobile Games":0, "MOBA":1, "FPS":2, "Battle Royale":3, "MMO":4, "RPG":5, "Strategy":6}
mapPrimary = {"Age of Empires":0, "Apex Legends":1, "CS:GO":2, "Call of Duty":3, "Candy Crush":4, "Civilization VI":5, 
              "Clash of Clans":6, "Cyberpunk 2077":7, "Dota 2":8, "Elden Ring":9, "Elder Scrolls Online":10, 
              "Final Fantasy XIV":11, "Fortnite":12, "Genshin Impact":13, "League of Legends":14, "Mobile Legends":15, 
              "Overwatch":16, "PUBG":17, "PUBG Mobile":18, "Skyrim":19, "StarCraft II":20, "Valorant":21, 
              "Warzone":22, "World of Warcraft":23 }
mapPlatform = {"PC":0, "Multi-platform":1, "Console":2, "Mobile":3}
mapSQual = {"Insomnia":0, "Very Poor":1, "Poor":2, "Fair":3, "Good":4}
mapSDis = {"Never":0, "Rarely":1, "Sometimes":2, "Often":3, "Always":4}
mapAperf = {"Failing":0, "Poor":1, "Below Average":2, "Average":3, "Good":4, "Excellent":5}
mapMood = {"Depressed":0, "Anxious":1, "Angry":2, "Irritable":3, "Withdrawn":4, "Restless":5, "Normal":6, "Excited":7, "Euphoric":8}
mapMSFreq = {"Never":0, "Rarely":1, "Sometimes":2, "Often":3, "Daily":4}
mapAddict = {"Low":0, "Moderate":1, "High":2, "Severe":3}
mapBinary = {"TRUE":1, "FALSE":0}
maps = [mapGender, mapGenre, mapPrimary, mapPlatform, mapSQual, mapSDis, mapAperf, mapMood, mapMSFreq, mapAddict]

In [5]:
def readData(datafile):
    # get the current directory: directory name for the abs path of the curr file
    dirpath = os.getcwd()
    abspath = dirpath + "\\" + datafile

    # read data into a pandas dataframe. data file is comma separated so use read_csv
    df = pd.read_csv(abspath)
     
    return df


# convert categorical data to numerical 
def convertData(data, drop:list[str], maps:list[dict], export:tuple[bool, str]):
    # drop indicated columns (should be a list of header names)
    if drop:
        for i in range(len(drop)):
            data = data.drop(columns=drop[i])

    if maps:
        idx=0
        for feat in data.columns:
            if data[feat].dtype == 'string':
                #print(f"Converting column {feat} with map {idx}")
                data[feat] = data[feat].map(maps[idx])
                idx += 1
            elif data[feat].dtype == 'bool':
                data[feat] = data[feat].astype(int)
                        
        if (export[0]):
            data.to_csv(export[1])

    return data

In [ ]:
filename = "GamingandMentalHealth.csv"  
data = readData(filename)

# drop primary game for now 
dropCols = ["record_id"]
exportTo = (True, "GamingandMentalHealth_conv_noid.csv")
data_conv = convertData(data.copy(), dropCols, maps, exportTo)

#print(data.head(1))

In [11]:
data = readData("GamingandMentalHealth_conv_noid.csv")
spearman = data.corr(method='spearman')
print(spearman)

                                  Unnamed: 0       age    gender  \
Unnamed: 0                          1.000000  0.018791  0.009253   
age                                 0.018791  1.000000  0.018518   
gender                              0.009253  0.018518  1.000000   
daily_gaming_hours                  0.005389 -0.013442 -0.028432   
game_genre                          0.029037 -0.012297 -0.011469   
primary_game                       -0.019954 -0.036285  0.032602   
gaming_platform                    -0.053698  0.022091  0.024227   
sleep_hours                         0.008475  0.020552  0.025449   
sleep_quality                       0.001145  0.052844  0.040515   
sleep_disruption_frequency          0.022526 -0.032659 -0.062881   
academic_work_performance           0.008431 -0.020869  0.030830   
grades_gpa                          0.002231 -0.023733  0.021369   
work_productivity_score            -0.008559 -0.033703  0.000349   
mood_state                          0.016457  0.

### Need to fill in missing data values before doing tSNE or anything else

In [12]:
data = data.to_numpy()
print(data)

x = data[:, 0:-1]       # data is all features
target = data[:,-1]     # target is mental_health_risk

scaler = StandardScaler()
xscaled = scaler.fit_transform(x)

ndims = 2 # dims of the embedded space
tsne = TSNE(ndims)
result = tsne.fit_transform(xscaled)
result_df = pd.DataFrame({'TSNE_col1': result[:,0], 'TSNE_col2': result[:,1], 'GamingMentalHealth' : target})
fig, ax = plt.subplots()
ax.scatter(x=result_df['TSNE_col1'], y=result_df['TSNE_col2'], c=result_df['GamingMentalHealth'])
plt.title("Gaming Mental health risk data in 2D")
plt.show()




[[  0.    17.     0.   ... 383.7    3.     3.  ]
 [  1.    21.     0.   ...  46.64   1.     0.  ]
 [  2.    23.     0.   ... 100.81   6.     3.  ]
 ...
 [997.    23.     0.   ...  88.6    5.     2.  ]
 [998.    18.     0.   ...  22.02   8.     0.  ]
 [999.    29.     0.   ...  25.2   19.     0.  ]]


ValueError: Input X contains NaN.
TSNE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values